# Talklate4u 用 PLaMo 3 Translate サーバー (Google Colab)

自分用・使うときだけ手動起動する前提の、OpenAI 互換翻訳サーバーです。

1. ランタイムのタイプを GPU に変更します(L4 推奨。T4 でも `DTYPE = 'float16'` で動く想定です)。
2. (任意)モデルのダウンロードで 401/403 になる場合のみ、左の鍵アイコン(シークレット)に `HF_TOKEN` を登録し、
   ノートブックからのアクセスを許可します。通常は不要です。
3. セルを上から順に実行します。最後に表示される **Base URL** と **API キー** を Talklate4u の設定に入力します。
4. 使い終わったら「停止」セルを実行し、ランタイムを切断してください(コンピュートユニット節約)。

Talklate4u 側の設定: LLM プロバイダー `openai` / モデル名 `plamo-3-translate-8b` / Base URL / API キー。

In [ ]:
# 設定
MODEL_ID = 'pfnet/plamo-3-translate-8b'  # Hugging Face のモデル ID(要確認)
SERVED_NAME = 'plamo-3-translate-8b'  # アプリ側のモデル名('plamo' を含めること)
DTYPE = 'auto'  # T4 の場合は 'float16'
MAX_MODEL_LEN = 4096
PORT = 8000

In [ ]:
# GPU 確認
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 依存関係のインストール (vLLM は plamo3 対応版 0.12 以降が必要)
!pip install -q -U 'vllm>=0.12.0'
!wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /content/cloudflared

In [ ]:
# 認証情報の準備 (API キー生成と、任意の HF トークン)
import os
import secrets

from google.colab import userdata

# HF_TOKEN は任意。登録されていれば使う(ゲート付きモデル用)。未登録でも続行する。
try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF_TOKEN を使用します。')
except Exception:
    print('HF_TOKEN 未登録のため、トークンなしで進めます。')
API_KEY = 'tk-' + secrets.token_urlsafe(24)
print('API キーを生成しました。(最後のセルで表示します)')

In [ ]:
# vLLM サーバーの起動 (OpenAI 互換 API)
import subprocess
import time
import urllib.request

vllm_log = open('/content/vllm.log', 'w')
vllm_proc = subprocess.Popen(
    [
        'vllm', 'serve', MODEL_ID,
        '--served-model-name', SERVED_NAME,
        '--trust-remote-code',
        '--dtype', DTYPE,
        '--max-model-len', str(MAX_MODEL_LEN),
        '--host', '127.0.0.1',
        '--port', str(PORT),
        '--api-key', API_KEY,
    ],
    stdout=vllm_log,
    stderr=subprocess.STDOUT,
)

deadline = time.time() + 900
while time.time() < deadline:
    if vllm_proc.poll() is not None:
        raise RuntimeError('vLLM が終了しました。/content/vllm.log を確認してください。')
    try:
        req = urllib.request.Request(
            f'http://127.0.0.1:{PORT}/v1/models', headers={'Authorization': f'Bearer {API_KEY}'}
        )
        urllib.request.urlopen(req, timeout=2)
        print('vLLM 起動完了')
        break
    except Exception:
        time.sleep(5)
else:
    raise TimeoutError('vLLM の起動がタイムアウトしました。/content/vllm.log を確認してください。')

In [ ]:
# cloudflared クイックトンネルの起動 (アカウント不要)
import re

tunnel_log = open('/content/cloudflared.log', 'w')
tunnel_proc = subprocess.Popen(
    ['/content/cloudflared', 'tunnel', '--no-autoupdate', '--url', f'http://127.0.0.1:{PORT}'],
    stdout=tunnel_log,
    stderr=subprocess.STDOUT,
)

public_url = None
for _ in range(60):
    time.sleep(1)
    with open('/content/cloudflared.log') as f:
        m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', f.read())
    if m:
        public_url = m.group(0)
        break

if not public_url:
    raise RuntimeError('トンネル URL を取得できませんでした。/content/cloudflared.log を確認してください。')

print('=== Talklate4u の設定 ===')
print('プロバイダー : openai')
print(f'モデル名     : {SERVED_NAME}')
print(f'Base URL     : {public_url}/v1')
print(f'API キー     : {API_KEY}')

In [ ]:
# 動作テスト (任意)
import json

body = json.dumps(
    {
        'model': SERVED_NAME,
        'messages': [{'role': 'user', 'content': 'Hello, nice to meet you.'}],
        'max_tokens': 64,
    }
).encode()
req = urllib.request.Request(
    f'{public_url}/v1/chat/completions',
    data=body,
    headers={'Content-Type': 'application/json', 'Authorization': f'Bearer {API_KEY}'},
)
print(urllib.request.urlopen(req, timeout=60).read().decode())

In [ ]:
# 停止 (使い終わったら実行し、ランタイムも切断すること)
tunnel_proc.terminate()
vllm_proc.terminate()